In [1]:
SEED, K = 42, 5
TRAIN_PATH = "../dont_get_kicked/datasets/training.csv"
TEST_PATH = "../dont_get_kicked/datasets/test.csv"
OUT_PATH = "xgb_kaggle_submission.csv"

DEVICE = "cpu"                                  # "cuda" tunes much faster on a Kaggle GPU
N_TRIALS, MAX_TREES, ES_ROUNDS = 100, 4000, 100
MIN_COUNT = 30                                  # rarer categorical levels are merged into "other"

## Data

In [6]:
import re
from collections import Counter
from typing import Optional

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

import json

import optuna
import xgboost as xgb

train, test = pd.read_csv(TRAIN_PATH), pd.read_csv(TEST_PATH)
train["PurchDate"], test["PurchDate"] = pd.to_datetime(train["PurchDate"]), pd.to_datetime(test["PurchDate"])
y = train["IsBadBuy"]
X, X_test, ref_id = train.drop(columns=["IsBadBuy", "RefId"]), test.drop(columns=["RefId"]), test["RefId"]
folds = list(StratifiedKFold(K, shuffle=True, random_state=SEED).split(X, y))
print(X.shape, X_test.shape, "positive rate:", round(y.mean(), 4))

(72983, 32) (48707, 32) positive rate: 0.123


## Feature engineering

In [7]:
MMR = ["MMRAcquisitionAuctionAveragePrice", "MMRAcquisitionAuctionCleanPrice", "MMRAcquisitionRetailAveragePrice",
       "MMRAcquisitonRetailCleanPrice", "MMRCurrentAuctionAveragePrice", "MMRCurrentAuctionCleanPrice",
       "MMRCurrentRetailAveragePrice", "MMRCurrentRetailCleanPrice"]
PAIRS = list(zip(MMR[:4], MMR[4:]))        # acquisition -> current price of the same type
CATS = ["Auction", "Make", "Model", "Trim", "SubModel", "Color", "Transmission", "WheelType", "Nationality", "Size",
        "TopThreeAmericanName", "PRIMEUNIT", "AUCGUART", "VNST", "BYRNO", "VNZIP1", "make_model"]

def make_features(df):
    f = {}
    date = pd.to_datetime(df["PurchDate"])
    f["buy_days"] = (date - pd.Timestamp("2009-01-01")).dt.days
    f["buy_month"], f["buy_dow"] = date.dt.month, date.dt.dayofweek
    for c in ["VehYear", "VehicleAge", "VehOdo", "WarrantyCost", "WheelTypeID", "IsOnlineSale"]:
        f[c] = df[c]
    f["odo_per_year"] = df["VehOdo"] / (df["VehicleAge"] + 1)

    price = df[MMR].where(df[MMR] > 100)                       # 0 / 1 / NaN are garbage, not prices
    cost = df["VehBCost"].where(df["VehBCost"] > 100)
    f["VehBCost"] = cost
    f.update({c: price[c] for c in MMR})
    f["warranty_to_cost"] = df["WarrantyCost"] / cost
    f["mmr_mean"], f["mmr_missing"] = price.mean(axis=1), price.isna().sum(axis=1)
    f["cost_to_mmr_mean"] = cost / f["mmr_mean"]
    f.update({"cost_to_" + c: cost / price[c] for c in MMR})
    f.update({"drift_" + b: np.log(price[b] / price[a]) for a, b in PAIRS})
    f["retail_to_auction_acq"] = price[MMR[3]] / price[MMR[1]]
    f["retail_to_auction_cur"] = price[MMR[7]] / price[MMR[5]]

    text = (df["Model"].fillna("") + " " + df["SubModel"].fillna("") + " " + df["Trim"].fillna("")).str.upper()
    f["doors"] = text.str.extract(r"(\d)D\b")[0].astype(float)
    f["is_4wd"] = text.str.contains(r"4WD|AWD|4X4").astype(int)
    f["liters"] = text.str.extract(r"(\d\.\d)L")[0].astype(float)
    f["make_model"] = df["Make"].fillna("") + "_" + df["Model"].fillna("")
    for c in CATS[:-1]:
        f[c] = df[c]
    return pd.DataFrame(f).replace([np.inf, -np.inf], np.nan)

def finalize(frames, cols, min_count):
    """Merge rare levels into "other" and give all frames the same category set (label-free, so train+test together)."""
    for c in cols:
        s = [f[c].astype(str).where(f[c].notna()) for f in frames]       # NaN stays NaN
        counts = pd.concat(s).value_counts()
        keep = counts.index[counts >= min_count]
        levels = sorted(set(keep) | {"other"})
        for f, v in zip(frames, s):
            f[c] = pd.Categorical(v.where(v.isin(keep) | v.isna(), "other"), categories=levels)
    return frames

F, F_test = finalize([make_features(X), make_features(X_test)], CATS, MIN_COUNT)
print("features:", F.shape[1], "| categorical:", sum(str(t) == "category" for t in F.dtypes))

features: 57 | categorical: 17


## Step 1: Optuna tuning of XGBoost on fold 0

In [8]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def make_model(params, n_trees, **extra):
    return xgb.XGBClassifier(n_estimators=n_trees, tree_method="hist", device=DEVICE, enable_categorical=True,
                             eval_metric="auc", random_state=SEED, **params, **extra)

tr, va = folds[0]
def objective(trial):
    p = {"learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
         "max_depth": trial.suggest_int("max_depth", 3, 10),
         "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 50.0, log=True),
         "subsample": trial.suggest_float("subsample", 0.5, 1.0),
         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
         "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
         "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
         "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
         "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 8),
         "max_cat_threshold": trial.suggest_int("max_cat_threshold", 8, 64)}
    m = make_model(p, MAX_TREES, early_stopping_rounds=ES_ROUNDS).fit(
        F.iloc[tr], y.iloc[tr], eval_set=[(F.iloc[va], y.iloc[va])], verbose=False)
    trees = int(m.best_iteration) + 1
    trial.set_user_attr("trees", trees)
    return roc_auc_score(y.iloc[va], m.predict_proba(F.iloc[va], iteration_range=(0, trees))[:, 1])

log = lambda study, t: print(f"trial {t.number + 1}: best valid AUC so far {study.best_value:.5f}") if (t.number + 1) % 10 == 0 else None
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, callbacks=[log])

BEST_PARAMS, BEST_TREES = study.best_trial.params, study.best_trial.user_attrs["trees"]
print("best valid AUC:", round(study.best_value, 5), "| trees:", BEST_TREES)
print(json.dumps(BEST_PARAMS, indent=2))

trial 10: best valid AUC so far 0.80109
trial 20: best valid AUC so far 0.80109
trial 30: best valid AUC so far 0.80338
trial 40: best valid AUC so far 0.80338
trial 50: best valid AUC so far 0.80338
trial 60: best valid AUC so far 0.80338
trial 70: best valid AUC so far 0.80338
trial 80: best valid AUC so far 0.80338
trial 90: best valid AUC so far 0.80338
trial 100: best valid AUC so far 0.80338
best valid AUC: 0.80338 | trees: 337
{
  "learning_rate": 0.023288234151743487,
  "max_depth": 10,
  "min_child_weight": 17.22789846602852,
  "subsample": 0.7981460114285993,
  "colsample_bytree": 0.39958712500533977,
  "gamma": 2.5788453538907707e-08,
  "reg_alpha": 1.3003732233906271e-08,
  "reg_lambda": 0.6647018484189298,
  "max_cat_to_onehot": 6,
  "max_cat_threshold": 50
}


## Step 2: 5-fold CV with the tuned parameters → submission

In [ ]:
oof, test_pred = np.zeros(len(X)), np.zeros(len(X_test))
for k, (tr, va) in enumerate(folds):
    model = make_model(BEST_PARAMS, BEST_TREES).fit(F.iloc[tr], y.iloc[tr])
    oof[va] = model.predict_proba(F.iloc[va])[:, 1]
    test_pred += model.predict_proba(F_test)[:, 1] / K
    print(f"fold {k}: val AUC={roc_auc_score(y.iloc[va], oof[va]):.5f}")
print("OOF ROC AUC:", round(roc_auc_score(y, oof), 5))
pd.DataFrame({"RefId": ref_id, "IsBadBuy": test_pred}).to_csv(OUT_PATH, index=False)
print("saved", OUT_PATH)

In [14]:
pd.DataFrame({'feature': F.columns, 'importance': model.feature_importances_}).sort_values('importance',ascending=False)

,feature,importance
7,WheelTypeID,0.299540
48,WheelType,0.162145
4,VehicleAge,0.035508
3,VehYear,0.030704
53,AUCGUART,0.025575
41,Auction,0.025378
52,PRIMEUNIT,0.022783
43,Model,0.022736
40,make_model,0.021072
56,VNZIP1,0.013838


In [37]:
DEFAULT_JDF_CAT_COLS = [c for c in F.columns if (F[c].dtype == 'category') or F[c].nunique() <13]
DEFAULT_JDF_NUM_COLS = list(set(F.columns) - set(DEFAULT_JDF_CAT_COLS))

In [39]:
class JointDistributionFilter(TransformerMixin):

    def __init__(self, X_train, X_test=None, num_cols=None, cat_cols=None, dp: float=0.02, th_num_train: float=0.001, th_num_test: float=0.001, th_cat_train: float=0.005, th_cat_test: float=0.005, min_dx_frac: float=0.001, bin_prefix: str='text__', verbose: bool=False):
        self.dp = dp
        self.th_num_train = th_num_train
        self.th_num_test = th_num_test
        self.th_cat_train = th_cat_train
        self.th_cat_test = th_cat_test
        self.min_dx_frac = min_dx_frac
        self.bin_prefix = bin_prefix
        self.verbose = verbose
        self.num_cols = list(num_cols) if num_cols is not None else list(DEFAULT_JDF_NUM_COLS)
        self.cat_cols = list(cat_cols) if cat_cols is not None else list(DEFAULT_JDF_CAT_COLS)
        self.train_profiles = {}
        self.test_profiles = {}
        self.bin_train_freq = {}
        self.bin_test_freq = {}
        self.bin_cols_to_drop_ = []
        self._build_profiles(X_train, is_test=False)
        if X_test is not None:
            self._build_profiles(X_test, is_test=True)
        self._build_bin_freqs(X_train, X_test)

    def _num_profile(self, s):
        x = pd.to_numeric(s, errors='coerce').dropna()
        n = len(x)
        if n == 0:
            return None
        shares = x.value_counts(normalize=True)
        heavy = shares[shares >= self.dp]
        heavy_shares = heavy.to_dict()
        rest = x[~x.isin(heavy.index)].values
        profile = {'heavy': heavy_shares, 'cx': None, 'log_cy': None}
        if len(rest) < 2:
            return profile
        rng = rest.max() - rest.min()
        if rng <= 0:
            return profile
        n_bins = int(round(1 / self.dp))
        edges = np.quantile(rest, np.linspace(0, 1, n_bins + 1))
        dx = np.maximum(np.diff(edges), self.min_dx_frac * rng)
        dens = 1.0 / n_bins / dx * rng
        centers = (edges[:-1] + edges[1:]) / 2.0
        cx = np.concatenate([[edges[0]], centers, [edges[-1]]])
        cy = np.concatenate([[dens[0]], dens, [dens[-1]]])
        profile['cx'], profile['log_cy'] = (cx, np.log(cy))
        return profile

    def _build_profiles(self, df, is_test: bool=False):
        profiles = {}
        for col in self.num_cols:
            if col in df.columns:
                p = self._num_profile(df[col])
                if p is not None:
                    profiles[col] = p
        for col in self.cat_cols:
            if col in df.columns:
                profiles[col] = df[col].value_counts(normalize=True).to_dict()
        if is_test:
            self.test_profiles = profiles
        else:
            self.train_profiles = profiles

    def _build_bin_freqs(self, X_train, X_test=None):
        bin_cols = [c for c in X_train.columns if str(c).startswith(self.bin_prefix)]
        self.bin_train_freq = {c: float(pd.to_numeric(X_train[c], errors='coerce').fillna(0).mean()) for c in bin_cols}
        self.bin_test_freq = {}
        if X_test is not None:
            self.bin_test_freq = {c: float(pd.to_numeric(X_test[c], errors='coerce').fillna(0).mean()) for c in bin_cols if c in X_test.columns}
        drop = []
        for c in bin_cols:
            if self.bin_train_freq[c] < self.th_cat_train:
                drop.append(c)
            elif X_test is not None and self.bin_test_freq.get(c, 0.0) < self.th_cat_test:
                drop.append(c)
        self.bin_cols_to_drop_ = drop
        if self.verbose:
            print(f'binary {self.bin_prefix}* columns: dropping {len(drop)} of {len(bin_cols)}')

    @staticmethod
    def _num_pass_mask(x, profile, th_num, th_heavy):
        if profile['cx'] is None:
            mask = np.zeros(len(x), dtype=bool)
        else:
            log_d = np.interp(x, profile['cx'], profile['log_cy'], left=-np.inf, right=-np.inf)
            mask = np.exp(log_d) >= th_num
        if profile['heavy']:
            share = pd.Series(x).map(profile['heavy']).values
            is_heavy = ~np.isnan(share)
            mask = np.where(is_heavy, np.nan_to_num(share) >= th_heavy, mask)
        return mask

    @staticmethod
    def _cat_share(values, profile):
        s = pd.Series(np.asarray(values, dtype=object))
        return s.map(profile).astype(float).fillna(0.0).values

    def _surviving_categories(self, col):
        cats = []
        test_profile = self.test_profiles.get(col)
        for k, share in self.train_profiles[col].items():
            if share < self.th_cat_train or share <= 0:
                continue
            if test_profile is not None and test_profile.get(k, 0.0) < self.th_cat_test:
                continue
            cats.append(k)
        return cats

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        for col in self.num_cols:
            if col not in X_out.columns or col not in self.train_profiles:
                continue
            x = pd.to_numeric(X_out[col], errors='coerce').astype(float).values
            final = self._num_pass_mask(x, self.train_profiles[col], self.th_num_train, self.th_cat_train)
            if col in self.test_profiles:
                final &= self._num_pass_mask(x, self.test_profiles[col], self.th_num_test, self.th_cat_test)
            was_nan = np.isnan(x)
            out = x.copy()
            out[~final] = np.nan
            X_out[col] = out
            if self.verbose:
                print(f'{col}: masked {(~final & ~was_nan).mean():.2%} of values')
        for col in self.cat_cols:
            if col not in X_out.columns or col not in self.train_profiles:
                continue
            vals = X_out[col].values
            final = self._cat_share(vals, self.train_profiles[col]) >= self.th_cat_train
            if col in self.test_profiles:
                final &= self._cat_share(vals, self.test_profiles[col]) >= self.th_cat_test
            cats = self._surviving_categories(col)
            X_out[col] = pd.Series(np.asarray(X_out[col].values, dtype=object), index=X_out.index).where(final).astype(pd.CategoricalDtype(categories=cats))
            if self.verbose:
                print(f'{col}: masked {(~final).mean():.2%} of values')
        drop_cols = [c for c in self.bin_cols_to_drop_ if c in X_out.columns]
        if drop_cols:
            X_out = X_out.drop(columns=drop_cols)
        return X_out

FX, FX_test = F.copy(), F_test.copy()
for c in DEFAULT_JDF_CAT_COLS:
    FX[c] = F[c].astype('category')
    FX_test = F_test[c].astype('category')
jdf = JointDistributionFilter(F, F_test, num_cols= DEFAULT_JDF_NUM_COLS, cat_cols= DEFAULT_JDF_CAT_COLS,
                              th_cat_train=0.002, th_cat_test=0.002)

FX, FX_test = jdf.transform(F), jdf.transform(F_test)

In [43]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def make_model(params,  **extra):
    return xgb.XGBClassifier( tree_method="hist", device=DEVICE, enable_categorical=True,
                             eval_metric="auc", random_state=SEED, **params, **extra)

def objective(trial):
    p = {"learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
         "max_depth": trial.suggest_int("max_depth", 5, 8),
         "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 50.0, log=True),
         "subsample": trial.suggest_float("subsample", 0.5, 1.0),
         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
         "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
         "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
         "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
         "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 8),
         "max_cat_threshold": trial.suggest_int("max_cat_threshold", 8, 64),
         "n_estimators" : trial.suggest_int("n_estimators", 300, 400)}
    oof_pred = np.zeros_like(y, dtype=np.float32)
    for tr, va in folds:
        m = make_model(p,).fit(FX.iloc[tr], y.iloc[tr], verbose=False)
        oof_pred[va] = m.predict_proba(FX.iloc[va])[:,1]
    return roc_auc_score(y, oof_pred)

N_TRIALS = 50
log = lambda study, t: print(f"trial {t.number + 1}: best valid AUC so far {study.best_value:.5f}") if (t.number + 1) % 1 == 0 else None
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, callbacks=[log])

BEST_PARAMS = study.best_trial.params
print("best valid AUC:", round(study.best_value, 5),)
print(json.dumps(BEST_PARAMS, indent=2))

trial 1: best valid AUC so far 0.79388
trial 2: best valid AUC so far 0.79388
trial 3: best valid AUC so far 0.79388
trial 4: best valid AUC so far 0.79388
trial 5: best valid AUC so far 0.79405
trial 6: best valid AUC so far 0.79405
trial 7: best valid AUC so far 0.79405
trial 8: best valid AUC so far 0.79405
trial 9: best valid AUC so far 0.79405
trial 10: best valid AUC so far 0.79405
trial 11: best valid AUC so far 0.79405
trial 12: best valid AUC so far 0.79540
trial 13: best valid AUC so far 0.79622
trial 14: best valid AUC so far 0.79622
trial 15: best valid AUC so far 0.79622
trial 16: best valid AUC so far 0.79622
trial 17: best valid AUC so far 0.79622
trial 18: best valid AUC so far 0.79622
trial 19: best valid AUC so far 0.79622
trial 20: best valid AUC so far 0.79622
trial 21: best valid AUC so far 0.79622
trial 22: best valid AUC so far 0.79622
trial 23: best valid AUC so far 0.79622
trial 24: best valid AUC so far 0.79622
trial 25: best valid AUC so far 0.79622
trial 26:

In [56]:
models = []
BEST_PARAMS["learning_rate"] = 0.003
BEST_PARAMS["n_estimators"] = 5000
for tr, va in folds:
    m = make_model(BEST_PARAMS,).fit(FX.iloc[tr], y.iloc[tr], verbose=False)
    models.append(m)


In [59]:
def f(n_est):
    oof_pred = np.zeros_like(y,dtype=np.float32)
    test_pred = np.zeros(len(FX_test))
    for i,(tr, va) in enumerate(folds):
        oof_pred[va] = models[i].predict_proba(FX.iloc[va],iteration_range=(0, n_est))[:,1]
        test_pred += models[i].predict_proba(FX_test,iteration_range=(0, n_est))[:,1]/len(folds)
    roc_auc_train_CV = roc_auc_score(y, oof_pred)
    return (roc_auc_train_CV,n_est, oof_pred, test_pred)


best = (-1,-1, None, None)
l,r = BEST_PARAMS["n_estimators"]//2,BEST_PARAMS["n_estimators"]
m = (l+r)//2
sl, sr, sm = f(l),f(r),f(m)
while True:
    lm , rm = (l+m)//2, (m+r)//2
    if (lm == l) or (rm==m):
        break
    if f(rm)[0]> f(lm)[0]:
        l = m
        m = rm
    else:
        r = m
        m = lm
    print((l,r))



(3750, 5000)
(4375, 5000)
(4687, 5000)
(4843, 5000)
(4921, 5000)
(4960, 5000)
(4980, 5000)
(4980, 4990)
(4980, 4985)
(4980, 4982)


In [60]:
q = f(l),f(r),f(m)
for a in q:
    print(a[0])

0.7977196772595645
0.7977195693446743
0.7977191411662389
